This notebook is a template for training models from a spectral library.

In [ ]:
# this cell can be used as a convenient way to remap the model path when executing the notebook directly

import os
os.environ['PHOENIX_MODEL_PATH'] = '../../phoenix/fullgrid'

In [ ]:
import time
import datetime
from pathlib import Path

import numpy as np

from matplotlib import pyplot as plt

import multiprocess

from astropy import units as u
from astropy import visualization as viz
viz.quantity_support()

from jwst import datamodels

from tqdm.auto import tqdm

from nirspectroperf.spec_model import BlackbodyResMLPModel, SpectrumModelBlackbodyResMLP,PhoenixModelsResampled, supress_multiple_slash_warning
supress_multiple_slash_warning()  # otherwise it fills the log for phoenix model loading

import jax
from jax import numpy as jnp
import jaxon
jax.devices()

In [ ]:
random_key_seed = 42  # can be None to use the time

spectrum_target_spacing = 0.5*u.angstrom
data_filename_slitnum = ('../../ngc6791_cals', 49)
spectrum_min_max_micron = None # this is set from min/maxing the example spectrum if it is None

cache_fn = 'spec_cache.pkl'  # None to not use the cache

device_dtype = jnp.float32

poolsize = None  # for parallelizing cpu-heavy operations

model_scaling = 0.5 # this is how the median-normalized spectra get rescaled.  0.5 makes most of the spectrum lie between 0 and 1

batch_size = 64
nepochs = 2500
ncycles = 4
minlr = 1e-4
peaklr = 1e-3
warmup_fraction = 0.1
validation_split_fraction = 0.2

save_model_fn_pattern = 'phoenix_estimator_{dt}.hdf5'  # use "{dt}" to get a timestamp

In [ ]:
if random_key_seed is None:
    random_key_seed = time.time_ns()
jrng = jax.random.key(random_key_seed)

# Load or Resample spectral data

In [ ]:
if data_filename_slitnum is None:
    if spectrum_min_max_micron is None:
        assert False, 'Need to specify either data_filename_slitnum or spectrum_min_max_micron'
else:
    n6791fits = list(Path(data_filename_slitnum[0]).glob('*.fits'))

    dm0 = datamodels.open(n6791fits[0])

    slit = dm0.slits[data_filename_slitnum[1]]
    swcs = slit.meta.wcs

    y, x = np.indices(slit.data.shape)
    sky, spec = swcs.pixel_to_world(x, y)
    densespec = np.linspace(np.nanmin(spec), np.nanmax(spec), x.shape[-1]*3)

    spectrum_min_max_micron = (np.min(densespec), np.max(densespec))
    
spectrum_min_max_micron

In [ ]:
resampled_spec_model = PhoenixModelsResampled.make_from_target_spacing(spectrum_target_spacing, *spectrum_min_max_micron)

In [ ]:
if cache_fn is not None:
    if Path(cache_fn).is_file():
        print('Found cache file', cache_fn, 'loading')
        resampled_spec_model.reload_cache(cache_fn)
    else:
        print('Cache file', cache_fn, 'not found, will regenderate')

        if poolsize is None:
            resampled_spec_model.make_all_spectra(None)
        else:
            with multiprocess.Pool(poolsize) as pool:
                resampled_spec_model.make_all_spectra(pool)

        resampled_spec_model.save_cache('spec_cache.pkl')

Ensure they match and that at least the first one seems reasonable

In [ ]:
np.testing.assert_almost_equal(resampled_spec_model.wave_microns[0], spectrum_min_max_micron[0].to_value(u.micron), decimal=4)
np.testing.assert_almost_equal(resampled_spec_model.wave_microns[-1], spectrum_min_max_micron[-1].to_value(u.micron), decimal=4)


fig, axs = plt.subplots(2, 1, sharex=True, figsize=(12, 8))
for idx, ax in zip([0, -1], axs):
    sparam = resampled_spec_model.get_available_teff_logg_feh()[idx]
    ax.plot(resampled_spec_model.wave_microns, resampled_spec_model.make_raw_spectrum(*sparam))
    ax.set_title(str(sparam))
axs[-1].set_xlabel(r'$\mu m$')

In [ ]:
stellar_params = []
stellar_models = []
for params in tqdm(resampled_spec_model.get_available_teff_logg_feh()):
    stellar_params.append(params)
    stellar_models.append(resampled_spec_model.make_raw_spectrum(*params))

stellar_params = jnp.array(stellar_params, dtype=device_dtype)
stellar_models = jnp.array(stellar_models, dtype=device_dtype)

We also do a simple normalization of scaling the models to their median times a scaling factor that can be set above as a (hyper) parameter

In [ ]:
normalized_stellar_models = (stellar_models.T/jnp.median(stellar_models,axis=1)).T * model_scaling

fig, axs = plt.subplots(3, 1, sharex=True, figsize=(12, 12))
for idx, ax in zip([0, 2000, -1], axs):
    sparam = resampled_spec_model.get_available_teff_logg_feh()[idx]
    ax.plot(resampled_spec_model.wave_microns, normalized_stellar_models[idx])
    ax.set_title(str(sparam))
axs[-1].set_xlabel(r'$\mu m$')

#min, ... median ..., max of all the spectra
jnp.quantile(normalized_stellar_models, jnp.array([0, 0.001, 0.01, .1, .5, .9, .99, .999, 1]))

# Train the model

In [ ]:
trainkwargs = {k:globals()[k] for k in ('batch_size', 'nepochs', 'validation_split_fraction', 'ncycles', 'minlr', 'peaklr', 'warmup_fraction')}

In [ ]:
modkey, jrng = jax.random.split(jrng)

model, training_info = SpectrumModelBlackbodyResMLP.train(stellar_parameters=stellar_params,
                                   normalized_stellar_models=normalized_stellar_models, 
                                   wave_microns=resampled_spec_model.wave_microns,
                                   random_key=modkey, 
                                   **trainkwargs)

In [ ]:
plt.semilogy(training_info['training_loss'],'.', ms=1, alpha=.4, label='Training Loss')
plt.semilogy(training_info['epoch_boundaries'], training_info['validation_loss'],'--', ms=2, alpha=.8, label='Validation Loss')
plt.legend()
plt.twinx()
plt.semilogy(training_info['learning_rates'], color='C2', alpha=.5, ls=':')
plt.ylabel('LR', color='C2')

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(16, 12))

vspr = training_info['validation_set']['stellar_parameters_rescaled']
vmods = training_info['validation_set']['normalized_stellar_models']
unscale = model.trained_model.stellar_parameters_rescaler.unscale_stellar_parameters

idxs = jax.random.choice(jax.random.key(42), vspr.shape[0], (axs.size,))

for ax, idx in zip(axs.ravel(), idxs):
    pred = model.trained_model(vspr[idx])
    ax.plot(model.wave_microns, pred.ravel(), label='Predicted')
    ax.plot(model.wave_microns, vmods[idx].ravel(), alpha=.5, label='True')
    bbpred = model.trained_model.bb_flamb_jax(model.wave_microns, unscale(vspr[idx])[0])
    ax.plot(model.wave_microns, bbpred.ravel(), label='Blackbody')
    ax.set_title(f"idx={idx}, Teff={unscale(vspr[idx])[0]:.0f}, logg={unscale(vspr[idx])[1]:.2f}, [M/H]={unscale(vspr[idx])[2]:.2f}")

axs.ravel()[0].legend(loc=0)
fig.tight_layout()
display(fig)

for ax in axs.ravel():
    ax.set_xlim(1.2,1.23)

# Save the trained model

In [ ]:
save_path = Path(save_model_fn_pattern.format(dt=datetime.datetime.now().isoformat().split('.')[0].replace(':','-')))
jaxon.save(save_path, model, allow_dill=True)
save_path.absolute(), save_path.stat().st_size/1024/1024 #MB

Load with something like:

```
model = jaxon.load(save_path, allow_dill=True)
```